In [1]:
import pandas as pd
import os
import numpy as np
import folium
import matplotlib.pyplot as plt

from matplotlib.backends.backend_pdf import PdfPages
from folium.plugins import FastMarkerCluster
from folium.plugins import MarkerCluster
from pathlib import Path
from datetime import datetime
from pyproj import Transformer

from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors

In [2]:
# Basisverzeichnis (aktuelles Notebook-Verzeichnis)
base_dir = Path.cwd()

# Pfad zu "Komplette_Datenbank" 
db_root = (
    base_dir.parent 
    / "1.1_Data-Acquisition-Wrapper"
    / "Angepasste_Datenbanken" / "Nach_Acquisition" / "Komplette_Datenbank"
)

# Alle Unterordner lesen, Ordner mit "neustem" Datum setzen
timestamp_folders = [f for f in db_root.iterdir() if f.is_dir()]
latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)

# Komplette Datenbank als CSV
database_path = latest_folder / "Komplette_Datenbank.csv"

print("Verwendeter Datenbankpfad:")
print(database_path)

Verwendeter Datenbankpfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Komplette_Datenbank\2025-11-27_12-13-25\Komplette_Datenbank.csv


In [3]:
# Ordner "Datenanalyse"
analysis_root = base_dir / "Rock-Type_Mapping"
analysis_root.mkdir(exist_ok=True)

# Zeitstempel erzeugen
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Neuen Ordner mit Zeitstempel erzeugen
output_dir = analysis_root / timestamp
output_dir.mkdir(parents=True, exist_ok=False)

print("Erzeugter Output-Ordner:")
print(output_dir)

Erzeugter Output-Ordner:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1.2_Rock-Type-Mapping\Rock-Type_Mapping\2025-11-27_12-22-40


<div class="alert alert-info">
    <b>Mapping von Lithology zu Rock-Type Spalte basierend auf</b><br>
    - Koordinaten der Datenbanken (Longitude und Latitude)<br>
    - GLiM Karte aus folgender Publikation: https://doi.pangaea.de/10.1594/PANGAEA.788537<br>
    -> Wird verwendet um Koordinaten auf Lithology global zu mappen
</div>

In [4]:
# Basisverzeichnis (aktuelles Notebook-Verzeichnis)
base_dir = Path.cwd()

# Pfad zu "glim_wgs84_0point5deg.txt.asc"
glim_wgs84_path = (
    base_dir
    / "GLiM-Kartierung"
    / "glim_wgs84_0point5deg.txt.asc"
)


print("Verwendeter Datenbankpfad:")
print(glim_wgs84_path)

Verwendeter Datenbankpfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1.2_Rock-Type-Mapping\GLiM-Kartierung\glim_wgs84_0point5deg.txt.asc


In [5]:
# Basisverzeichnis (aktuelles Notebook-Verzeichnis)
base_dir = Path.cwd()

# Pfad zu "glim_wgs84_0point5deg.txt.asc"
finalized_database_with_lithology = (
    output_dir
    / "finalized_database_with_lithology.csv"
)

print("Verwendeter Datenbankpfad:")
print(finalized_database_with_lithology)

Verwendeter Datenbankpfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1.2_Rock-Type-Mapping\Rock-Type_Mapping\2025-11-27_12-22-40\finalized_database_with_lithology.csv


In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
import rasterio

def add_glim_rock_type(glim_wgs84_path, database_path):

    # Rock Type Spalte komplett ersetzen und in neuem Ordner speichern
    # So kann anschließend überprüft werden, ob die bereits vorhandenen Daten übereinstimmen

    glim_wgs84_path = Path(glim_wgs84_path)
    database_path = Path(database_path)

    # ----- Direkte Mapping-Tabelle: Raster-Code -> voller String
    code_to_full = {
        1:  "unconsolidated sediments",
        2:  "basic volcanic rocks",
        3:  "siliciclastic sedimentary rocks",
        4:  "basic plutonic rocks",
        5:  "mixed sedimentary rocks",
        6:  "carbonate sedimentary rocks",
        7:  "acid volcanic rocks",
        8:  "metamorphic rocks",
        9:  "acid plutonic rocks",
        10: "intermediate volcanic rocks",
        11: "water bodies",
        12: "pyroclastic rocks",
        13: "intermediate plutonic rocks",
        14: "evaporites",
        15: "no data",
        16: "ice and glaciers",
    }


    # ------- Datenbank als String einlesen ----------
    df = pd.read_csv(database_path, dtype=str, low_memory=False)


    # ------- rock_type-Spalte sicherstellen ---------
    if "rock_type" not in df.columns:
        df["rock_type"] = np.nan

    # ------- Koordinaten temporär numerisch ----------
    lat_num = pd.to_numeric(df["WGS84_latitude"], errors="coerce")
    lon_num = pd.to_numeric(df["WGS84_longitude"], errors="coerce")

    # -------- GLiM-Raster ------------
    with rasterio.open(glim_wgs84_path) as src:
        glim_data = src.read(1)   # ------ erstes Band
        nodata = src.nodata

        def get_lithology_full(lat, lon):
            # ----- Fehlende Koordinaten
            if pd.isna(lat) or pd.isna(lon):
                return np.nan
            try:
                # ------- rasterio: (x, y) = (lon, lat)
                row, col = src.index(lon, lat)
            except Exception:
                return np.nan

            # ------------ Bounds check
            if (row < 0 or row >= glim_data.shape[0] or
                col < 0 or col >= glim_data.shape[1]):
                return np.nan

            code = glim_data[row, col]

            # -------------- NoData Behandlung
            if nodata is not None and code == nodata:
                return np.nan

            # -------------- Ganzzahl sicherstellen
            try:
                code_int = int(code)
            except (TypeError, ValueError):
                return np.nan

            # --------------- Direkt auf vollen String mappen
            return code_to_full.get(code_int, np.nan)

        # ------- Lithologie für alle Zeilen ----------
        new_rock_types = [
            get_lithology_full(lat_num.iloc[i], lon_num.iloc[i])
            for i in range(len(df))
        ]
        df["rock_type"] = pd.Series(new_rock_types, dtype="string")

    # ------ Ergebnis speichern -----------
    import pathlib
    _p = pathlib.Path(finalized_database_with_lithology)
    if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(finalized_database_with_lithology, index=False)

    print(f"Fertig! Datei gespeichert unter:\n{finalized_database_with_lithology}")
    return finalized_database_with_lithology

In [7]:
add_glim_rock_type(glim_wgs84_path, database_path)

Fertig! Datei gespeichert unter:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1.2_Rock-Type-Mapping\Rock-Type_Mapping\2025-11-27_12-22-40\finalized_database_with_lithology.csv


WindowsPath('C:/Users/lucca/OneDrive/SPEICHER/Hochschule/7. Semester/Abschlussarbeit Bearbeitung/Jupyter Notebooks/1.2_Rock-Type-Mapping/Rock-Type_Mapping/2025-11-27_12-22-40/finalized_database_with_lithology.csv')

<div class="alert alert-info">
    <b>In Excel konvertieren, Kopie aus 1.1</b><br>
</div>

In [8]:
def csv_to_excel(csv_path: Path, excel_path: Path, sheet_name: str = "SHEET1") -> None:

    # ------------------------------ CSV einlesen ---------------------------------
    df = pd.read_csv(csv_path, low_memory=False)

    # ------------------------- In Excel schreiben --------------------------------
    excel_path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
        import pathlib
        _p = pathlib.Path(writer)
        if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
        df.to_excel(writer, index=False, sheet_name=sheet_name)

        # ------------------- Zugriff auf Arbeitsmappe -----------------------------
        workbook = writer.book
        worksheet = writer.sheets[sheet_name]

        
        # ---------------------- Kopfzeilen-Format --------------------------------
        header_format = workbook.add_format({
            "bold": True,
            "text_wrap": True,
            "valign": "center",
            "fg_color": "#D3D3D3",
            "border": 1
        })
        for col_num, value in enumerate(df.columns.values):
            worksheet.write(0, col_num, value, header_format)

        
        # ---------------------- Spaltenbreite dynamisch ---------------------------
        for i, col in enumerate(df.columns):
            max_len = max(df[col].astype(str).map(len).max(), len(col)) + 2
            worksheet.set_column(i, i, min(max_len, 50))  # begrenze max. Breite

    print(f"Excel-Pfad:\n{excel_path.resolve()}")

In [9]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
finalized_database_with_lithology
excel_path = (
    output_dir
    / "finalized_database_with_lithology.xlsx"
)

In [10]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
import pathlib
_p = pathlib.Path(finalized_database_with_lithology)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(finalized_database_with_lithology, excel_path)


Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1.2_Rock-Type-Mapping\Rock-Type_Mapping\2025-11-27_12-22-40\finalized_database_with_lithology.xlsx
